# Experiment 1 — Class-weight ablation
## XGBoost + GLCM + kernelRadius=5 + cùng patient folds

Mục tiêu: chỉ thay đổi **cơ chế trọng số lớp**:
- `none`: không truyền `sample_weight`;
- `balanced`: tính `compute_class_weight("balanced")` từ **training fold** rồi truyền `sample_weight`.

Các yếu tố còn lại giữ cố định: Development, patient folds, GLCM, XGBoost config và threshold = 0.50.

**Không tải Test trong notebook này.**

In [3]:
import os, json
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, balanced_accuracy_score
)

RANDOM_STATE = 42
N_SPLITS = 5

# Giữ đúng đường dẫn của notebook hiện tại.
# Nếu dữ liệu thực tế của bạn nằm ở dataset/577p/... thì chỉ sửa biến này.
DATASET_PATH = "dataset/577p/glcm/kernel5/"
TRAIN_PATH = os.path.join(DATASET_PATH, "dataset.train.csv")
TEST_PATH = "dataset/1200p/glcm/kernel5/dataset.test.csv"

BASE_CV_OUTPUT = "output/577p/glcm/kernel5/patient_5fold_cv_saeed_comparison"

XGB_CONFIG = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    min_child_weight=1,
    gamma=0.0,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

OUTPUT_DIR = "output/577p/glcm/kernel5/class_weight_ablation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cấu hình OOF chính thức dùng nối tiếp cho Experiment 2 và 3.
# Phải đồng bộ với WEIGHTING_MODE trong notebook 03.
OFFICIAL_OOF_MODE = "balanced"  # "balanced" hoặc "none"
OFFICIAL_OOF_PATH = os.path.join(OUTPUT_DIR, "development_oof_predictions.csv")

print("TEST is not loaded in this experiment.")
print("Official OOF mode:", OFFICIAL_OOF_MODE)
print("Official OOF output:", OFFICIAL_OOF_PATH)


TEST is not loaded in this experiment.
Official OOF mode: balanced
Official OOF output: output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv


## 1. Load Development

In [4]:
train_df = pd.read_csv(TRAIN_PATH)

development = pd.concat([
    train_df.assign(original_split="train"),
], ignore_index=True)

PATIENT_CANDIDATES = ["patient_id", "patient", "subject_id", "case_id"]
patient_col = next((c for c in PATIENT_CANDIDATES if c in development.columns), None)

if patient_col is None:
    raise ValueError("Development CSV thiếu patient_id/subject_id/case_id; không được tiếp tục patient-level CV.")
if "label" not in development.columns:
    raise ValueError("Development CSV thiếu cột label.")

development[patient_col] = development[patient_col].astype(str)
train_df[patient_col] = train_df[patient_col].astype(str)

META = {
    "label", "original_split", patient_col,
    "patient_id", "patient", "subject_id", "case_id",
    "x", "y", "z", "i", "j", "k", "voxel_x", "voxel_y", "voxel_z"
}
numeric_cols = [
    c for c in development.columns
    if c not in META and pd.api.types.is_numeric_dtype(development[c])
]
feature_cols = [c for c in numeric_cols]
if not feature_cols:
    raise ValueError("Không tìm thấy cột GLCM trong Development.")

print("Development shape:", development.shape)
print("Development patients:", development[patient_col].nunique())
print("GLCM features:", len(feature_cols))

Development shape: (653397, 27)
Development patients: 577
GLCM features: 24


## 2. Khóa patient folds

Ưu tiên sử dụng `patient_fold_assignment.csv` do notebook CV hiện tại đã tạo.
Nếu file này chưa có, notebook tái tạo folds bằng **GroupKFold**, đúng với notebook hiện tại.

In [5]:
assignment_path = os.path.join(BASE_CV_OUTPUT, "patient_fold_assignment.csv")

if os.path.exists(assignment_path):
    fold_assignment = pd.read_csv(assignment_path)
    if patient_col not in fold_assignment.columns or "validation_fold" not in fold_assignment.columns:
        raise ValueError("patient_fold_assignment.csv không có cột cần thiết.")
    fold_assignment[patient_col] = fold_assignment[patient_col].astype(str)
    print("Loaded existing fold assignment:", assignment_path)
else:
    X_tmp = development[feature_cols]
    y_tmp = development["label"].astype(int)
    g_tmp = development[patient_col].astype(str)
    gkf = GroupKFold(n_splits=N_SPLITS)
    rows = []
    for fold, (_, va_idx) in enumerate(gkf.split(X_tmp, y_tmp, g_tmp), 1):
        for pid in sorted(g_tmp.iloc[va_idx].unique()):
            rows.append({patient_col: pid, "validation_fold": fold})
    fold_assignment = pd.DataFrame(rows)
    fold_assignment.to_csv(os.path.join(OUTPUT_DIR, "patient_fold_assignment_generated.csv"), index=False)
    print("Existing assignment not found; generated folds with GroupKFold.")

if fold_assignment[patient_col].duplicated().any():
    raise RuntimeError("Một patient xuất hiện ở validation của nhiều hơn một fold.")
if set(fold_assignment[patient_col]) != set(development[patient_col].unique()):
    missing = set(development[patient_col].unique()) - set(fold_assignment[patient_col])
    extra = set(fold_assignment[patient_col]) - set(development[patient_col].unique())
    raise RuntimeError(f"Fold assignment không khớp Development. missing={len(missing)}, extra={len(extra)}")

patient_to_fold = dict(zip(fold_assignment[patient_col], fold_assignment["validation_fold"].astype(int)))
development["validation_fold"] = development[patient_col].map(patient_to_fold)
print(development.groupby("validation_fold")[patient_col].nunique())

Existing assignment not found; generated folds with GroupKFold.
validation_fold
1    115
2    115
3    116
4    116
5    115
Name: patient_id, dtype: int64


## 3. Run two weighting conditions

In [6]:
def binary_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
        "roc_auc": roc_auc_score(y_true, prob) if np.unique(y_true).size == 2 else np.nan,
        "pr_auc": average_precision_score(y_true, prob) if np.unique(y_true).size == 2 else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

In [7]:
conditions = ["none", "balanced"]
if OFFICIAL_OOF_MODE not in conditions:
    raise ValueError("OFFICIAL_OOF_MODE phải là 'balanced' hoặc 'none'.")

fold_rows = []
oof_parts = []

for condition in conditions:
    for fold in range(1, N_SPLITS + 1):
        va_mask = development["validation_fold"].eq(fold)
        tr_mask = ~va_mask

        tr = development.loc[tr_mask].copy()
        va = development.loc[va_mask].copy()

        tr_pat = set(tr[patient_col].unique())
        va_pat = set(va[patient_col].unique())
        if tr_pat & va_pat:
            raise RuntimeError(f"{condition}, fold {fold}: patient leakage")

        Xtr = tr[feature_cols]
        ytr = tr["label"].astype(int)
        Xva = va[feature_cols]
        yva = va["label"].astype(int)

        fit_kwargs = {}
        w0 = w1 = np.nan

        if condition == "balanced":
            classes = np.array([0, 1])
            weights = compute_class_weight(class_weight="balanced", classes=classes, y=ytr)
            cw = dict(zip(classes, weights))
            fit_kwargs["sample_weight"] = ytr.map(cw).to_numpy()
            w0, w1 = float(cw[0]), float(cw[1])

        model = XGBClassifier(**XGB_CONFIG)
        model.fit(Xtr, ytr, verbose=False, **fit_kwargs)

        prob = model.predict_proba(Xva)[:, 1]
        m = binary_metrics(yva, prob, threshold=0.5)

        fold_rows.append({
            "condition": condition,
            "fold": fold,
            "train_patients": tr[patient_col].nunique(),
            "validation_patients": va[patient_col].nunique(),
            "train_voxels": len(tr),
            "validation_voxels": len(va),
            "class_weight_0": w0,
            "class_weight_1": w1,
            **m
        })

        part = va[[patient_col, "label"]].copy()
        for c in ["x","y","z","i","j","k","voxel_x","voxel_y","voxel_z"]:
            if c in va.columns:
                part[c] = va[c].values
        part["condition"] = condition
        part["fold"] = fold
        part["prob_tumor"] = prob
        part["pred_t05"] = (prob >= 0.5).astype(int)
        oof_parts.append(part)

        print(condition, "fold", fold, "F1", round(m["f1"], 4), "PR-AUC", round(m["pr_auc"], 4))

fold_results = pd.DataFrame(fold_rows)
oof = pd.concat(oof_parts, ignore_index=True)

# Lưu toàn bộ kết quả ablation (cả none và balanced).
fold_results.to_csv(
    os.path.join(OUTPUT_DIR, "class_weight_ablation_fold_metrics.csv"),
    index=False
)
oof.to_csv(
    os.path.join(OUTPUT_DIR, "class_weight_ablation_oof_predictions.csv"),
    index=False
)

# Lưu riêng OOF theo từng condition để audit/reuse khi cần.
for condition in conditions:
    condition_oof = oof.loc[oof["condition"].eq(condition)].copy()
    condition_oof.to_csv(
        os.path.join(OUTPUT_DIR, f"development_oof_predictions_{condition}.csv"),
        index=False
    )

# Xuất OOF CHÍNH THỨC dùng cho notebook 02 và 03.
official_oof = oof.loc[oof["condition"].eq(OFFICIAL_OOF_MODE)].copy()

# Kiểm tra mỗi voxel Development chỉ xuất hiện đúng một lần trong OOF chính thức.
if len(official_oof) != len(development):
    raise RuntimeError(
        f"Official OOF rows ({len(official_oof)}) != Development rows ({len(development)})."
    )

if official_oof[patient_col].nunique() != development[patient_col].nunique():
    raise RuntimeError(
        "Official OOF không bao phủ đúng toàn bộ bệnh nhân Development."
    )

expected_patients = set(development[patient_col].astype(str).unique())
oof_patients = set(official_oof[patient_col].astype(str).unique())
if expected_patients != oof_patients:
    raise RuntimeError(
        f"Official OOF patient mismatch: missing={len(expected_patients - oof_patients)}, "
        f"extra={len(oof_patients - expected_patients)}"
    )

official_oof.to_csv(OFFICIAL_OOF_PATH, index=False)

print("\nSaved official Development OOF:")
print("  mode:", OFFICIAL_OOF_MODE)
print("  path:", OFFICIAL_OOF_PATH)
print("  rows:", len(official_oof))
print("  patients:", official_oof[patient_col].nunique())
display(fold_results)


none fold 1 F1 0.7239 PR-AUC 0.8273
none fold 2 F1 0.686 PR-AUC 0.7732
none fold 3 F1 0.7153 PR-AUC 0.7943
none fold 4 F1 0.7436 PR-AUC 0.8206
none fold 5 F1 0.6933 PR-AUC 0.783
balanced fold 1 F1 0.6345 PR-AUC 0.8216
balanced fold 2 F1 0.5738 PR-AUC 0.764
balanced fold 3 F1 0.5834 PR-AUC 0.7923
balanced fold 4 F1 0.5304 PR-AUC 0.8115
balanced fold 5 F1 0.579 PR-AUC 0.7712

Saved official Development OOF:
  mode: balanced
  path: output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv
  rows: 653397
  patients: 577


,condition,fold,train_patients,validation_patients,train_voxels,validation_voxels,class_weight_0,class_weight_1,accuracy,balanced_accuracy,...,recall,specificity,f1,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,none,1,462,115,522979,130418,NaN,NaN,0.969498,0.809961,...,0.627104,0.992817,0.723903,0.717785,0.974062,0.827254,121225,877,3101,5215
1,none,2,462,115,522996,130401,NaN,NaN,0.964011,0.801410,...,0.614969,0.987851,0.686024,0.672266,0.967501,0.773177,120581,1483,3210,5127
2,none,3,461,116,522306,131091,NaN,NaN,0.969334,0.816250,...,0.642294,0.990205,0.715338,0.704453,0.965462,0.794296,122020,1207,2813,5051
3,none,4,461,116,522320,131077,NaN,NaN,0.967416,0.850286,...,0.715325,0.985247,0.743622,0.726902,0.971440,0.820638,120612,1806,2465,6194
4,none,5,462,115,522987,130410,NaN,NaN,0.964627,0.803863,...,0.619268,0.988458,0.693264,0.680261,0.969279,0.782988,120584,1408,3205,5213
5,balanced,1,462,115,522979,130418,0.533978,7.857729,0.934235,0.916075,...,0.895262,0.936889,0.634508,0.634523,0.973618,0.821602,114396,7706,871,7445
6,balanced,2,462,115,522996,130401,0.533954,7.862946,0.915047,0.905506,...,0.894566,0.916445,0.573825,0.579857,0.966790,0.763984,111865,10199,879,7458
7,balanced,3,461,116,522306,131091,0.534519,7.742455,0.923671,0.908389,...,0.891022,0.925755,0.583430,0.590045,0.966050,0.792337,114078,9149,857,7007
8,balanced,4,461,116,522320,131077,0.533649,7.929558,0.892094,0.906226,...,0.922508,0.889943,0.530412,0.545371,0.970689,0.811493,108945,13473,671,7988
9,balanced,5,462,115,522987,130410,0.533866,7.882008,0.915789,0.907101,...,0.897125,0.917077,0.579008,0.584609,0.968887,0.771172,111876,10116,866,7552


### OOF output dùng cho Experiment 2 và 3

`development_oof_predictions.csv` chỉ chứa OOF của cấu hình `OFFICIAL_OOF_MODE` đã khóa. Mỗi bệnh nhân Development chỉ được dự đoán ở fold mà bệnh nhân đó thuộc validation, vì vậy file này có thể dùng tiếp cho per-patient bootstrap và chọn threshold mà không dùng Test.


## 4. Mean ± SD and paired fold deltas

In [8]:
metric_cols = [
    "accuracy","balanced_accuracy","precision","recall","specificity",
    "f1","mcc","roc_auc","pr_auc"
]

summary_rows = []
for condition, g in fold_results.groupby("condition"):
    row = {"condition": condition}
    for metric in metric_cols:
        row[f"{metric}_mean"] = g[metric].mean()
        row[f"{metric}_sd"] = g[metric].std(ddof=1)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUTPUT_DIR, "class_weight_ablation_summary.csv"), index=False)

wide = fold_results.pivot(index="fold", columns="condition", values=metric_cols)
delta_rows = []
for fold in sorted(fold_results["fold"].unique()):
    row = {"fold": fold}
    for metric in metric_cols:
        row[f"delta_balanced_minus_none_{metric}"] = (
            wide.loc[fold, (metric, "balanced")] - wide.loc[fold, (metric, "none")]
        )
    delta_rows.append(row)

paired_delta = pd.DataFrame(delta_rows)
paired_delta.to_csv(os.path.join(OUTPUT_DIR, "class_weight_ablation_paired_deltas.csv"), index=False)

display(summary)
display(paired_delta)

,condition,accuracy_mean,accuracy_sd,balanced_accuracy_mean,balanced_accuracy_sd,precision_mean,precision_sd,recall_mean,recall_sd,specificity_mean,specificity_sd,f1_mean,f1_sd,mcc_mean,mcc_sd,roc_auc_mean,roc_auc_sd,pr_auc_mean,pr_auc_sd
0,balanced,0.916167,0.015521,0.908659,0.004283,0.429425,0.042381,0.900097,0.012722,0.917222,0.017352,0.580237,0.037016,0.586881,0.031849,0.969207,0.003065,0.792118,0.024875
1,none,0.966977,0.002570,0.816354,0.019822,0.800081,0.033941,0.643792,0.041321,0.988916,0.002815,0.712430,0.023333,0.700334,0.023549,0.969549,0.003351,0.799671,0.023503


,fold,delta_balanced_minus_none_accuracy,delta_balanced_minus_none_balanced_accuracy,delta_balanced_minus_none_precision,delta_balanced_minus_none_recall,delta_balanced_minus_none_specificity,delta_balanced_minus_none_f1,delta_balanced_minus_none_mcc,delta_balanced_minus_none_roc_auc,delta_balanced_minus_none_pr_auc
0,1,-0.035264,0.106115,-0.364654,0.268158,-0.055929,-0.089395,-0.083262,-0.000444,-0.005652
1,2,-0.048964,0.104096,-0.353261,0.279597,-0.071405,-0.112199,-0.092409,-0.000711,-0.009193
2,3,-0.045663,0.092139,-0.373418,0.248728,-0.064450,-0.131907,-0.114408,0.000587,-0.001958
3,4,-0.075322,0.055939,-0.402040,0.207183,-0.095305,-0.213210,-0.181532,-0.000751,-0.009144
4,5,-0.048838,0.103238,-0.359904,0.277857,-0.071382,-0.114256,-0.095652,-0.000392,-0.011816


## Interpretation rule

Không chọn cấu hình bằng Test. Nếu cần khóa weighting cho final model, lựa chọn phải dựa trên kết quả Development/CV.
Nên ưu tiên PR-AUC và F1-score, đồng thời theo dõi Precision, Recall và MCC.